# E3: RL training of the view-selection policy (Colab)
Inputs: the framework zip (with the `viewplan` files) and `to_colab_*.zip` from `bash run_all.sh build`.
Outputs: `to_jetson_*.zip` (policies) and `results_views.zip` (training results for the paper).
Every reward comes from images the robot actually took; nothing here is simulated.

In [ ]:
from google.colab import files
up = files.upload()   # framework zip + to_colab_*.zip

In [ ]:
import glob, os
fw = [f for f in up if not f.startswith('to_colab')][0]
!unzip -oq "$fw" -d /content
root = sorted(glob.glob('/content/*/views.py'))[0].rsplit('/', 1)[0]
%cd $root
for z in glob.glob('/content/to_colab_*.zip'):
    !unzip -oq "$z" -d .
import numpy as np
z = np.load('data/view_library.npz')
print(len(z['names']), 'placements', dict(zip(*np.unique(z['poses'], return_counts=True))), '| test:', list(z['names'][z['split'] == 1]))

In [ ]:
# transformer and MLP ablation, 3 seeds each (~5-15 min per run on CPU)
for arch in ['attn', 'mlp']:
    for s in [0, 1, 2]:
        !python train_views.py --arch {arch} --seed {s} --iters 2000

In [ ]:
# input ablations of the transformer
for s in [0, 1, 2]:
    !python train_views.py --arch attn --seed {s} --iters 2000 --no-pose
    !python train_views.py --arch attn --seed {s} --iters 2000 --no-time

In [ ]:
# deploy the seed with the best VALIDATION score (never chosen on test results)
import numpy as np, shutil
for arch in ['attn', 'mlp']:
    best = max([0, 1, 2], key=lambda s: float(np.load(f'deploy/view_policy_{arch}_s{s}.npz')['info_val_pct']))
    shutil.copy(f'deploy/view_policy_{arch}_s{best}.npz', f'deploy/view_policy_{arch}.npz')
    print(arch, '-> seed', best)
!python analyze_views.py
!python plot_all.py

In [ ]:
!python sync.py pack-for-jetson
!zip -qr results_views.zip results_views paper/figs
files.download(sorted(glob.glob('to_jetson_*.zip'))[-1]); files.download('results_views.zip')